In [1]:
import pandas as pd
import numpy as np

df_tow = pd.read_parquet("dane/interim/towar_columns_selected-records_full.parquet")
df_asort = pd.read_parquet("dane/interim/asort_columns_selected-records_full.parquet")
df_dok = pd.read_parquet("dane/interim/dok_columns_selected-records_full.parquet")
df_pozd = pd.read_parquet("dane/interim/pozdok_columns_selected-records_full.parquet")

print(df_tow.shape, df_asort.shape, df_dok.shape, df_pozd.shape)

(41378, 11) (337, 2) (952122, 9) (4272656, 11)


In [2]:
# TODO: opcjonalnie po wersji MVP, jako jedna z poprawek - aktualizacja tabeli towarów (więcej SKU)
# df_tow = pd.read_csv(
#     "dane/Towar_2026.csv",
#     encoding='utf-8-sig',  # obsługuje BOM
#     sep=';',
#     decimal=",",
#     on_bad_lines='skip'
# )
# print(df_towar_new.shape)
# print(df_towar_new['TowId'].max())
# print(df_towar_new[df_towar_new['TowId'] > 81360]['TowId'].nunique())
# df_towar_new[df_towar_new['TowId']>81360]['Nazwa']

In [3]:
print("df_tow TowId:", df_tow['TowId'].dtype)
print("df_dok DokId:", df_dok['DokId'].dtype)
print("df_pozd TowId:", df_pozd['TowId'].dtype)
print("df_pozd DokId:", df_pozd['DokId'].dtype)
print("df_asort AsId:", df_asort['AsId'].dtype)
print("df_tow AsId:", df_tow['AsId'].dtype)

df_tow TowId: int64
df_dok DokId: int64
df_pozd TowId: int64
df_pozd DokId: int64
df_asort AsId: int64
df_tow AsId: int64


In [4]:
# Krok 1: PozDok + Dok
dok_pozd = df_pozd.merge(df_dok, on='DokId', how='inner')
print(f"dok_pozd: {dok_pozd.shape}")
# Krok 2: + Towar
dok_pozd_tow = dok_pozd.merge(df_tow, on='TowId', how='inner')
print(f"dok_pozd_tow: {dok_pozd_tow.shape}")
# Krok 3: + Asort
fact_inka = dok_pozd_tow.merge(df_asort, on='AsId', how='inner')

print(f"fact_inka: {fact_inka.shape}")
print(fact_inka.columns.tolist())

dok_pozd: (4272656, 19)
dok_pozd_tow: (4261678, 29)
fact_inka: (4261678, 30)
['DokId', 'Kolejnosc', 'NrPozycji', 'TowId', 'TypPoz', 'IloscPlus', 'IloscMinus', 'CenaPrzedRab', 'CenaPoRab', 'Wartosc', 'CenaDet', 'Data', 'KolejnyWDniu', 'NrDok', 'TypDok', 'AktywnyDok', 'Razem', 'DoZaplaty', 'Zaplacono', 'AsId', 'JMId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'Marza', 'Stawka', 'AktywnyTow', 'CleanName', 'NazwaAsort']


In [5]:
doc_type_map = pd.DataFrame([
    (2,   'PZ',             1, True,  'IP',           +1, 'przyjecie', False),
    (8,   'ZWPAR',          1, True,  'IP',           +1, 'zwrot',     False),
    (9,   'PW',             1, True,  'IP',           +1, 'przyjecie', False),  # ← NOWY jak PZ
   
    (10,  'RW',             4, True,  'IP',           -1, 'rozchod',   False),
    (21,  'DF',             4, True,  'IP',           -1, 'sprzedaz',  False),
    (23,  'ST',             4, True,  'IP',           -1, 'strata',    True),
    
    (14,  'BO',             3, True,  'RESET_IP',     +1, 'bo',        False),
    (16,  'REM',            3, True,  'RESET_IP',     +1, 'remanent',  False),

    (26,  'ROZB',           1, True,  'IP_IM_DELTA',  +1, 'rozbieznosc',      False),
    (78,  'PRZES',          1, True,  'IP_IM_DELTA',  +1, 'przesuniecie',     False),

    (88,  'PRZES_GR',       2, True,  'IP_IM_DELTA',  +1, 'przesuniecie',     False),
    
    (1,   'OPAK',           2, False, 'Brak',         0, 'neutralny', False),
    (4,   'ZWFD',           4, False, 'Brak',         0, 'neutralny', False),
    (18,  'PRZEC',          6, False, 'Brak',         0, 'przecena',  True),
    (19,  'ZAMR_PRZEC',     0, False, 'Brak',         0, 'przecena',  True),
    (30,  'B_OPAK',         0, False, 'Brak',         0, 'neutralny', False),
    (33,  'FV',             0, False, 'Brak',         0, 'neutralny', False),
    (50,  'ZAM',            0, False, 'Brak',         0, 'neutralny', False),
    (59,  'DF',             0, False, 'Brak',         0, 'neutralny', False),
    (60,  'XXX',            0, False, 'Brak',         0, 'neutralny', False),
    (81,  'CDPRS',          1, False, 'Brak',         0, 'prasa',     False),
    (82,  'CRPRS',          7, False, 'Brak',         0, 'prasa',     False),
    (100, 'ZM_ST_VAT',      1, False, 'Brak',         0, 'neutralny', False),
    (100, 'ZM_ST_VAT',      4, False, 'Brak',         0, 'neutralny', False),
    (126, 'PLAN_ZM_ST_VAT', 0, False, 'Brak',         0, 'neutralny', False),
    (900, 'ST_rozli',       4, False, 'Brak',         0, 'neutralny', False),
    (900, 'ZWPAR_rozli',    1, False, 'Brak',         0, 'neutralny', False),
    (981, 'MM_INW_MOB',     0, False, 'Brak',         0, 'neutralny', False),  # ← NOWY
], columns=['TypDok', 'Dokument', 'TypPoz', 'WplywNaStan',
            'MetodaLiczenia', 'Mnoznik', 'TypRuchu', 'CzyNiechciane'])

In [6]:
fact_inka_merge = fact_inka.merge(doc_type_map, on=['TypDok', 'TypPoz'], how='left')

In [7]:
fact_inka_merge_data = fact_inka_merge[fact_inka_merge['Data'].dt.year.isin([2023, 2024, 2025, 2026])]
print(f"Po filtrze dat: {fact_inka_merge_data.shape}")

Po filtrze dat: (4260543, 36)


In [8]:
fact_inka_merge_data.to_parquet("dane/interim/fact_inka_records_2023_2026.parquet", compression='zstd',index=False)
print(f"Zapisano: {fact_inka_merge_data.shape}")

Zapisano: (4260543, 36)


In [9]:
fact_inka_merge.to_parquet("dane/interim/fact_inka-records_full.parquet", index=False)
print(f"Zapisano: {fact_inka_merge.shape}")

Zapisano: (4261678, 36)
